# 🔧 Notebook 02 — Feature Engineering & SMOTE Balancing
**Financial Fraud Detection System**

This notebook:
1. Engineers four domain-informed features from raw transaction data
2. Demonstrates SMOTE resampling to solve the class imbalance
3. Saves the train/test split for model training

In [ ]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 120})

In [ ]:
df = pd.read_csv("../data/cleaned_data.csv")
print(f"Loaded cleaned data: {df.shape}")
df.head()

## 1. Feature Engineering

### Feature 1 & 2 — Time-based features

In [ ]:
# Time (seconds) → hour of day (0–23)
df["hour_of_day"]    = (df["Time"] // 3600 % 24).astype(int)
df["high_risk_hour"] = df["hour_of_day"].apply(lambda h: 1 if (h >= 23 or h <= 4) else 0)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Fraud count by hour
fraud_by_hour = df[df["Class"] == 1].groupby("hour_of_day").size()
axes[0].bar(fraud_by_hour.index, fraud_by_hour.values, color="#C0392B", alpha=0.8)
axes[0].set_title("Fraud Count by Hour of Day", fontweight="bold")
axes[0].set_xlabel("Hour")
axes[0].set_ylabel("Fraud Count")
axes[0].axvspan(-0.5, 4.5,  alpha=0.15, color="red", label="High-risk hours")
axes[0].axvspan(22.5, 23.5, alpha=0.15, color="red")
axes[0].legend()

# High-risk hour flag
hr_counts = df[df["Class"] == 1].groupby("high_risk_hour").size()
axes[1].pie(hr_counts.values, labels=["Normal Hours", "High-Risk Hours (11pm–4am)"],
            autopct="%1.1f%%", colors=["#27AE60", "#C0392B"], startangle=90)
axes[1].set_title("Fraud: High-Risk vs. Normal Hours", fontweight="bold")

plt.tight_layout()
plt.savefig("../outputs/graphs/02_time_features.png", bbox_inches="tight")
plt.show()

print(f"High-risk hour transactions (fraud): {hr_counts.get(1, 0)} / {hr_counts.sum()}")

### Feature 3 & 4 — Amount-based features

In [ ]:
df["amount_zscore"]  = df["Amount_Scaled"]          # Already standardised; z-score = same
high_value_threshold = df["Amount_Scaled"].quantile(0.99)
df["high_value_flag"] = (df["Amount_Scaled"] > high_value_threshold).astype(int)

print(f"High-value threshold (99th percentile): z-score = {high_value_threshold:.4f}")
print(f"High-value transactions: {df['high_value_flag'].sum():,}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Z-score distribution by class
for cls, label, colour in [(0, "Legitimate", "#27AE60"), (1, "Fraud", "#C0392B")]:
    subset = df[df["Class"] == cls]["amount_zscore"]
    axes[0].hist(subset.clip(-3, 10), bins=60, alpha=0.6, label=label, color=colour)
axes[0].set_title("Amount Z-Score Distribution by Class", fontweight="bold")
axes[0].set_xlabel("Z-Score")
axes[0].set_ylabel("Frequency")
axes[0].legend()

# High-value flag vs fraud
cross = pd.crosstab(df["high_value_flag"], df["Class"])
cross.plot(kind="bar", ax=axes[1], color=["#27AE60", "#C0392B"],
           edgecolor="white", width=0.6)
axes[1].set_title("High-Value Flag vs Fraud", fontweight="bold")
axes[1].set_xlabel("High-Value Flag (0=No, 1=Yes)")
axes[1].set_ylabel("Count")
axes[1].legend(["Legitimate", "Fraud"])
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig("../outputs/graphs/02_amount_features.png", bbox_inches="tight")
plt.show()

### Feature 5 & 6 — PCA interaction features

In [ ]:
df["V1_V3_interact"]  = df["V1"] * df["V3"]
df["V4_V11_interact"] = df["V4"] * df["V11"]

print("Created interaction features:")
print(f"  V1_V3_interact  — corr with Class: {df['V1_V3_interact'].corr(df['Class']):.4f}")
print(f"  V4_V11_interact — corr with Class: {df['V4_V11_interact'].corr(df['Class']):.4f}")

# Drop Time after extraction
df = df.drop(columns=["Time"], errors="ignore")
print(f"\nFinal feature count: {df.shape[1] - 1} features + 1 target")

## 2. SMOTE — Solving Class Imbalance

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Before SMOTE — Train set:")
print(f"  Legitimate: {(y_train == 0).sum():,}")
print(f"  Fraud:      {(y_train == 1).sum():,}")

# Apply SMOTE
sm = SMOTE(random_state=42, sampling_strategy=1.0)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

print(f"\nAfter SMOTE — Train set:")
print(f"  Legitimate: {(y_train_res == 0).sum():,}")
print(f"  Fraud:      {(y_train_res == 1).sum():,}")
print(f"\nTest set (UNCHANGED — no SMOTE applied to test data):")
print(f"  Legitimate: {(y_test == 0).sum():,}")
print(f"  Fraud:      {(y_test == 1).sum():,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, y_data, title in [
    (axes[0], y_train, "Before SMOTE (Training)"),
    (axes[1], y_train_res, "After SMOTE (Training)"),
]:
    counts = y_data.value_counts()
    bars = ax.bar(["Legitimate", "Fraud"], counts.values,
                  color=["#27AE60", "#C0392B"], edgecolor="white", width=0.5)
    ax.bar_label(bars, labels=[f"{v:,}" for v in counts.values],
                 padding=5, fontweight="bold")
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Count")

plt.suptitle("Effect of SMOTE on Training Data Class Balance",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../outputs/graphs/02_smote_comparison.png", bbox_inches="tight")
plt.show()

## 3. Final Feature Summary

In [ ]:
new_features = ["hour_of_day", "high_risk_hour", "amount_zscore",
                "high_value_flag", "V1_V3_interact", "V4_V11_interact"]

feat_corr = df[new_features + ["Class"]].corr()["Class"].drop("Class").abs()
feat_corr = feat_corr.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(feat_corr.index, feat_corr.values, color="#3498DB", edgecolor="white")
ax.bar_label(bars, fmt="%.4f", padding=3, fontsize=9)
ax.set_title("New Feature Correlation with Fraud (|r|)", fontweight="bold")
ax.set_xlabel("|Pearson r|")
plt.tight_layout()
plt.savefig("../outputs/graphs/02_feature_correlation.png", bbox_inches="tight")
plt.show()

print("\nEngineered Features Summary:")
print("-" * 45)
for f in new_features:
    print(f"  {f:<22} corr={feat_corr.get(f, 0):.4f}")

## Summary

| Feature | Type | Rationale |
|---|---|---|
| `hour_of_day` | Ordinal (0–23) | Time pattern differs between fraud/legit |
| `high_risk_hour` | Binary | Late-night flag — peak fraud period |
| `amount_zscore` | Continuous | Deviation from typical spend |
| `high_value_flag` | Binary | Top 1% amount — high-risk transactions |
| `V1_V3_interact` | Continuous | PCA interaction signal |
| `V4_V11_interact` | Continuous | PCA interaction signal |

**SMOTE insight:** Training set balanced to 50/50 — model will no longer be biased
toward always predicting "Legitimate".

**Next step:** `03_model_building.ipynb` — train, tune and evaluate classifiers